In [7]:
import requests

url = "https://www.matweb.com/search/DataSheet.aspx?MatGUID=ef57c33963404798ad0301a05692312a"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
html = response.text

print("Status:", response.status_code)
print("Final URL:", response.url)
print("Title-ähnlich:", response.text.split("</title>")[0][-200:])  # grob
print("Erste 400 Zeichen:\n", response.text[:400])


Status: 403
Final URL: https://www.matweb.com/search/DataSheet.aspx?MatGUID=ef57c33963404798ad0301a05692312a
Title-ähnlich: <!DOCTYPE html><html lang="en-US"><head><title>Just a moment...
Erste 400 Zeichen:
 <!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjus


In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC

url = "https://www.matweb.com/search/DataSheet.aspx?MatGUID=ef57c33963404798ad0301a05692312a"

opts = Options()
opts.add_experimental_option("debuggerAddress", "127.0.0.1:9222")

driver = webdriver.Edge(options=opts)  # <-- ATTACH, kein neuer Browser

driver.get(url)

WebDriverWait(driver, 90).until_not(EC.title_contains("Just a moment"))
WebDriverWait(driver, 60).until(
    EC.presence_of_element_located((By.XPATH, "//*[contains(., 'Physical Properties')]"))
)

html = driver.page_source
tables = pd.read_html(html)
print("Tabellen:", len(tables))

# NICHT driver.quit() !!!

Tabellen: 10


In [11]:
for i, t in enumerate(tables):
    print(f"\n===== Tabelle {i} =====")
    print("Shape:", t.shape)
    print("Columns:", list(t.columns))
    print(t.head(50))




===== Tabelle 0 =====
Shape: (0, 1)
Columns: ['System Message']
Empty DataFrame
Columns: [System Message]
Index: []

===== Tabelle 1 =====
Shape: (3, 3)
Columns: [0, 1, 2]
    0                                                  1  \
0 NaN                             Advertise with MatWeb!   
1 NaN  Data sheets for over 185,000 metals, plastics,...   
2 NaN  HOME • SEARCH • TOOLS • SUPPLIERS • FOLDERS • ...   

                                                   2  
0                                                NaN  
1  Data sheets for over 185,000 metals, plastics,...  
2  HOME • SEARCH • TOOLS • SUPPLIERS • FOLDERS • ...  

===== Tabelle 2 =====
Shape: (1, 2)
Columns: [0, 1]
                                                0   1
0  Recently Viewed Materials (most recent at top) NaN

===== Tabelle 3 =====
Shape: (1, 1)
Columns: [0]
                                                   0
0  Login to see your most recently viewed materia...

===== Tabelle 4 =====
Shape: (3, 2)
Columns: [0,

In [5]:
selected_df = None

for i, t in enumerate(tables):
    if 0 in t.columns:
        if t[0].astype(str).str.contains("Physical Properties", case=False, na=False).any():
            print(f"Gefundene Tabelle: {i}")
            
            # Nur Spalte 0 und 1 behalten (alles ab Spalte 2 abschneiden)
            selected_df = t.iloc[:, :2].copy()
            
            break

if selected_df is not None:
    print("Shape:", selected_df.shape)
    display(selected_df.head(50))
else:
    print("Keine passende Tabelle gefunden.")


Gefundene Tabelle: 7
Shape: (106, 2)


,0,1
0,NaN,NaN
1,Physical Properties,Metric
2,Density,10.22 g/cc
3,a Lattice Constant,3.147 Å @Temperature 25.0 °C
4,Vapor Pressure,0.00133 bar @Temperature 3102 °C
5,NaN,0.0133 bar @Temperature 3555 °C
6,NaN,0.133 bar @Temperature 4109 °C
7,NaN,0.267 bar @Temperature 4322 °C
8,NaN,0.533 bar @Temperature 4553 °C
9,NaN,1.01 bar @Temperature 4804 °C


In [4]:
import pandas as pd
import re

def extract_to_wide_lists(df_2cols: pd.DataFrame, material_name: str = "X") -> pd.DataFrame:
    df = df_2cols.iloc[:, :2].copy()
    df.columns = ["prop_raw", "metric_raw"]

    # Eigenschaft nach unten füllen
    df["prop"] = df["prop_raw"].ffill()

    # Überschriften/Leerzeilen raus
    drop_props = {
        "Physical Properties", "Chemical Properties", "Mechanical Properties", "Electrical Properties", "Metric"
    }
    df = df[df["metric_raw"].notna() & df["prop"].notna()]
    df = df[~df["prop"].isin(drop_props)]
    df = df[df["metric_raw"].astype(str).str.strip().ne("Metric")]

    # Zahl-Regex (inkl. scientific notation)
    num_re = re.compile(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?")

    def parse_metric(text: str):
        """Return (value, unit, temp_value, temp_unit). value/temp_value floats or None."""
        s = str(text).strip()

        # Temperatur abtrennen
        temp_val, temp_unit = None, None
        if "@Temperature" in s:
            left, right = s.split("@Temperature", 1)
            s = left.strip()

            mtemp = num_re.search(right)
            if mtemp:
                temp_val = float(mtemp.group())
                rest = right[mtemp.end():].strip()
                # Temperatureinheit (erstes Token), z.B. "°C"
                temp_unit = rest.split()[0] if rest else None

        # Wert + Einheit
        mval = num_re.search(s)
        if not mval:
            return None, None, temp_val, temp_unit

        value = float(mval.group())
        unit = s[mval.end():].strip()
        unit = unit if unit else None

        return value, unit, temp_val, temp_unit

    # Sammelcontainer: prop_key -> dict(values=[...], temps=[...], unit=..., tunit=...)
    store = {}

    for _, r in df.iterrows():
        prop = str(r["prop"]).strip()
        value, unit, tval, tunit = parse_metric(r["metric_raw"])

        if prop not in store:
            store[prop] = {"values": [], "temps": [], "unit": unit, "tunit": tunit}

        # falls später eine Zeile mit Einheit kommt und zuvor None war: auffüllen
        if store[prop]["unit"] is None and unit is not None:
            store[prop]["unit"] = unit
        if store[prop]["tunit"] is None and tunit is not None:
            store[prop]["tunit"] = tunit

        if value is not None:
            store[prop]["values"].append(value)
        if tval is not None:
            store[prop]["temps"].append(tval)

    # Wide-Output bauen
    out = {}
    for prop, d in store.items():
        unit = d["unit"]
        tunit = d["tunit"]

        val_col = f"{prop} ({unit})" if unit else prop
        temp_col = None
        if len(d["temps"]) > 0:
            temp_col = f"Temperature {prop} ({tunit})" if tunit else f"Temperature {prop}"

        # Werte: 1 -> scalar, >1 -> Liste
        vals = d["values"]
        temps = d["temps"]

        if len(vals) == 0:
            out[val_col] = None
        elif len(vals) == 1:
            out[val_col] = vals[0]
        else:
            out[val_col] = vals  # Liste

        if temp_col is not None:
            if len(temps) == 1:
                out[temp_col] = temps[0]
            else:
                out[temp_col] = temps  # Liste

    wide = pd.DataFrame([out], index=[material_name])
    wide.index.name = "Material"
    return wide


# Anwendung:
new_df = extract_to_wide_lists(selected_df, material_name="X")
display(new_df)

new_df.to_csv("material_X.csv")

,Density (g/cc),a Lattice Constant (Å),Temperature a Lattice Constant (°C),Vapor Pressure (bar),Temperature Vapor Pressure (°C),Atomic Mass,Atomic Number,Atomic Volume,Thermal Neutron Cross Section (barns/atom),X-ray Absorption Edge (Å),...,Melting Point (°C),Boiling Point (°C),Heat of Formation (kJ/mol),Emissivity (0-1),Temperature Emissivity (0-1) (°C),"Reflection Coefficient, Visible (0-1)","Molybdenum, Mo (%)",Descriptive Properties,CAS Number (-98-7),"Solubility (O, diluted Acid and Alkaline)"
Material,,,,,,,,,,,,,,,,,,,,,
X,10.22,3.147,25.0,"[0.00133, 0.0133, 0.133, 0.267, 0.533, 1.01]","[3102.0, 3555.0, 4109.0, 4322.0, 4553.0, 4804.0]",95.94,42.0,1.530000e-29,2.5,"[0.61977, 4.32066, 4.7133, 4.9093]",...,2617.0,4639.0,"[0.0, 658.1]",0.17,100.0,0.46,100.0,None,7439.0,2.0


In [16]:
# =========================
# COMPLETE PROGRAM
# =========================
# Requirements:
# - selenium
# - pandas
# - lxml (for pd.read_html)
# - Edge browser + Selenium Manager (webdriver.Edge) available
#
# What it does:
# 1) For each material, tries a list of URLs until one loads (Cloudflare "Just a moment" handled)
# 2) Extracts HTML tables with pd.read_html
# 3) Finds the table that contains "Physical Properties" in column 0
# 4) Cuts to the first two columns (0 and 1)
# 5) Extracts properties into ONE ROW per material:
#    - Values stored as scalar if single, list if multiple
#    - Temperatures stored in a separate column, scalar or list
# 6) Appends each material as a new row to a master dataframe
#    - New properties become new columns at the end
# 7) Writes master dataframe to CSV

import pandas as pd
import re

from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# -------------------------
# 1) Selenium: load one of many URLs, return tables
# -------------------------
def get_tables_from_urls(urls, wait_title_sec=90, wait_pp_sec=60):
    """
    Opens ONE browser, tries URLs one by one.
    Returns (tables, used_url). If nothing works: ([], None)
    """
    if not urls:
        return [], None

    opts = Options()
    opts.add_argument("--window-size=1400,900")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    # not headless due to Cloudflare

    driver = webdriver.Edge(options=opts)

    try:
        for url in urls:
            try:
                print(f"Versuche URL: {url}")
                driver.get(url)

                # wait until Cloudflare interstitial is gone
                WebDriverWait(driver, wait_title_sec).until_not(
                    EC.title_contains("Just a moment")
                )

                # wait until page contains "Physical Properties"
                WebDriverWait(driver, wait_pp_sec).until(
                    EC.presence_of_element_located(
                        (By.XPATH, "//*[contains(., 'Physical Properties')]")
                    )
                )

                html = driver.page_source
                tables = pd.read_html(html)

                if tables:
                    print(f"✅ Erfolgreich geladen: {url}  | Tabellen: {len(tables)}")
                    return tables, url
                else:
                    print(f"⚠️ Keine Tabellen gefunden: {url}")

            except Exception as e:
                print(f"❌ Fehlgeschlagen: {url} | {type(e).__name__}: {e}")
                continue

        print("Keine URL konnte erfolgreich geladen werden.")
        return [], None

    finally:
        driver.quit()


# -------------------------
# 2) Find the "Physical Properties" table and cut to 2 cols
# -------------------------
def select_physical_properties_table(tables):
    """
    Returns a 2-column DataFrame (cols 0 and 1) for the table that contains
    'Physical Properties' in column 0. If not found, returns None.
    """
    for t in tables:
        if isinstance(t, pd.DataFrame) and t.shape[1] >= 2:
            col0 = t.iloc[:, 0].astype(str)
            if col0.str.contains("Physical Properties", case=False, na=False).any():
                return t.iloc[:, :2].copy()
    return None


# -------------------------
# 3) Extract properties -> ONE ROW with lists for multi values
# -------------------------
def extract_to_wide_lists(df_2cols: pd.DataFrame, material_name: str = "X") -> pd.DataFrame:
    df = df_2cols.iloc[:, :2].copy()
    df.columns = ["prop_raw", "metric_raw"]

    # forward fill property names (so NaN rows belong to the property above)
    df["prop"] = df["prop_raw"].ffill()

    # remove section headers and empty rows
    drop_props = {
        "Physical Properties", "Chemical Properties", "Mechanical Properties", "Electrical Properties", "Metric"
    }
    df = df[df["metric_raw"].notna() & df["prop"].notna()]
    df = df[~df["prop"].isin(drop_props)]
    df = df[df["metric_raw"].astype(str).str.strip().ne("Metric")]

    # number regex (including scientific notation)
    num_re = re.compile(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?")

    def parse_metric(text: str):
        """
        Return (value, unit, temp_value, temp_unit)
        value/temp_value floats or None
        """
        s = str(text).strip()

        # split temperature part
        temp_val, temp_unit = None, None
        if "@Temperature" in s:
            left, right = s.split("@Temperature", 1)
            s = left.strip()

            mtemp = num_re.search(right)
            if mtemp:
                temp_val = float(mtemp.group())
                rest = right[mtemp.end():].strip()
                temp_unit = rest.split()[0] if rest else None  # e.g. °C

        # parse value + unit
        mval = num_re.search(s)
        if not mval:
            return None, None, temp_val, temp_unit

        value = float(mval.group())
        unit = s[mval.end():].strip()
        unit = unit if unit else None

        return value, unit, temp_val, temp_unit

    # store per property
    store = {}
    for _, r in df.iterrows():
        prop = str(r["prop"]).strip()
        value, unit, tval, tunit = parse_metric(r["metric_raw"])

        if prop not in store:
            store[prop] = {"values": [], "temps": [], "unit": unit, "tunit": tunit}

        # fill missing unit info if later found
        if store[prop]["unit"] is None and unit is not None:
            store[prop]["unit"] = unit
        if store[prop]["tunit"] is None and tunit is not None:
            store[prop]["tunit"] = tunit

        if value is not None:
            store[prop]["values"].append(value)
        if tval is not None:
            store[prop]["temps"].append(tval)

    # build wide row
    out = {}
    for prop, d in store.items():
        unit = d["unit"]
        tunit = d["tunit"]

        val_col = f"{prop} ({unit})" if unit else prop

        vals = d["values"]
        if len(vals) == 0:
            out[val_col] = None
        elif len(vals) == 1:
            out[val_col] = vals[0]
        else:
            out[val_col] = vals  # list

        temps = d["temps"]
        if len(temps) > 0:
            temp_col = f"Temperature {prop} ({tunit})" if tunit else f"Temperature {prop}"
            out[temp_col] = temps[0] if len(temps) == 1 else temps

    wide = pd.DataFrame([out], index=[material_name])
    wide.index.name = "Material"
    return wide


# -------------------------
# 4) Append material row to master DF (new props become new columns at the end)
# -------------------------
def add_material(master_df: pd.DataFrame | None, row_df: pd.DataFrame) -> pd.DataFrame:
    """
    master_df: existing master table (rows=materials, cols=properties)
    row_df: one-row dataframe from extract_to_wide_lists (index=material name)
    """
    if master_df is None or master_df.empty:
        return row_df

    combined = pd.concat([master_df, row_df], axis=0, sort=False)

    # keep existing column order, append new columns at end
    new_cols = [c for c in row_df.columns if c not in master_df.columns]
    combined = combined[list(master_df.columns) + new_cols]

    return combined


# -------------------------
# 5) Main pipeline: materials -> master_df
# -------------------------
def build_master(materials):
    """
    materials = [
      {"name": "Mo", "urls": ["https://...", "https://..."]},
      {"name": "W",  "urls": ["https://...", "https://..."]},
    ]
    """
    master_df = None

    for m in materials:
        name = m.get("name", "X")
        urls = m.get("urls", [])

        tables, used_url = get_tables_from_urls(urls)
        if not tables:
            print(f"[WARN] {name}: keine Tabellen (übersprungen).")
            continue

        selected_df = select_physical_properties_table(tables)
        if selected_df is None:
            print(f"[WARN] {name}: 'Physical Properties' Tabelle nicht gefunden (übersprungen).")
            continue

        row = extract_to_wide_lists(selected_df, material_name=name)
        master_df = add_material(master_df, row)

        print(f"[OK] {name}: hinzugefügt. (Quelle: {used_url})")

    return master_df if master_df is not None else pd.DataFrame()


# -------------------------
# 6) EXAMPLE USAGE
# -------------------------
if __name__ == "__main__":
    # Replace with your real URLs (each material can have multiple fallbacks)
    materials = [
        {"name": "Mo", "urls": ["https://www.matweb.com/search/DataSheet.aspx?MatGUID=ef57c33963404798ad0301a05692312a", "https://www.matweb.com/search/DataSheet.aspx?MatGUID=ef57c33963404798ad0301a05692312a"]},
        {"name": "W", "urls": ["https://www.matweb.com/search/DataSheet.aspx?MatGUID=41e0851d2f3c417ba69ea0188fa570e3", "https://www.matweb.com/search/DataSheet.aspx?MatGUID=41e0851d2f3c417ba69ea0188fa570e3"]},
    ]

    master_df = build_master(materials)

    print("\n===== MASTER DF =====")
    print(master_df.shape)
    print(master_df.head())

    # Save (use utf-8 for °C and Å)
    master_df.to_csv("materials_master.csv", encoding="utf-8", sep=";")
    print("\n✅ Gespeichert als materials_master.csv")

Versuche URL: https://www.matweb.com/search/DataSheet.aspx?MatGUID=ef57c33963404798ad0301a05692312a
❌ Fehlgeschlagen: https://www.matweb.com/search/DataSheet.aspx?MatGUID=ef57c33963404798ad0301a05692312a | TypeError: argument of type 'NoneType' is not iterable
Versuche URL: https://www.matweb.com/search/DataSheet.aspx?MatGUID=ef57c33963404798ad0301a05692312a
❌ Fehlgeschlagen: https://www.matweb.com/search/DataSheet.aspx?MatGUID=ef57c33963404798ad0301a05692312a | NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: MicrosoftEdge=145.0.3800.70)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7b6656815
	0x7ff7b6656874
	0x7ff7b699ad8a
	0x7ff7b638da9d
	0x7ff7b6418d96
	0x7ff7b642eabf
	0x7ff7b63e9a1c
	0x7ff7b63e8c76
	0x7ff7b63e9843
	0x7ff7b64b0474
	0x7ff7b64ac913
	0x7ff7b64bcfd3
	0x7ff7b6670ef8
	0x7ff7b66797d6
	0x7ff7b665de54
	0x7ff7b665df99
	0x7ff7b664bdd2
	0x7ffe073ae8d7
	0x7ffe089ac40c

K